# This Section Concerns all mentioned Datasets with log_reg

# Imports

In [30]:
# Standard library
import os

# Data handling
import numpy as np
import pandas as pd
from scipy import sparse

# ML utilities

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from cuml.linear_model import LogisticRegression as cuLR
from cuml.preprocessing import StandardScaler

from cuml.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score


def to_gpu_dense(X):
    """
    Converts input matrix X into a dense CuPy array safely.
    - If X is SciPy CSR/CSC/COO sparse → convert to NumPy dense → CuPy
    - If X is pandas DataFrame → convert to NumPy → CuPy
    - If X is NumPy array → CuPy
    - Never return nested object-arrays
    """

    # Case 1: SciPy sparse (CSR, CSC, COO)
    if sparse.issparse(X):
        # Convert sparse → dense NumPy → CuPy
        X_np = X.toarray().astype(np.float32)
        return cp.asarray(X_np)

    # Case 2: Pandas DataFrame
    if hasattr(X, "values"):
        return cp.asarray(X.values.astype(np.float32))

    # Case 3: NumPy array
    if isinstance(X, np.ndarray):
        return cp.asarray(X.astype(np.float32))

    # If it's already CuPy
    if isinstance(X, cp.ndarray):
        return X

    raise TypeError(f"Unsupported type passed to to_gpu_dense(): {type(X)}")

def load_split(data, split, base_path="../data/splits/"):
    """
    Loads X_train, X_test, y_train, y_test for a given dataset + split.
    Automatically detects whether features are stored as sparse (.npz)
    or dense (.csv).

    Example:
        X_train, X_test, y_train, y_test = load_split("cup98", "7030")
    """

    path = os.path.join(base_path, split)

    # ---- Load X_train ----
    npz_path = os.path.join(path, f"X_train_{data}.npz")
    csv_path = os.path.join(path, f"X_train_{data}.csv")

    if os.path.exists(npz_path):
        X_train = sparse.load_npz(npz_path)
    else:
        X_train = pd.read_csv(csv_path)

    # ---- Load X_test ----
    npz_path = os.path.join(path, f"X_test_{data}.npz")
    csv_path = os.path.join(path, f"X_test_{data}.csv")

    if os.path.exists(npz_path):
        X_test = sparse.load_npz(npz_path)
    else:
        X_test = pd.read_csv(csv_path)

    # ---- Load labels ----
    y_train = pd.read_csv(os.path.join(path, f"y_train_{data}.csv"))
    y_test  = pd.read_csv(os.path.join(path, f"y_test_{data}.csv"))

    # Convert DataFrames → Series
    y_train = y_train.iloc[:, 0]
    y_test  = y_test.iloc[:, 0]

    return X_train, X_test, y_train, y_test



def load_all_by_split(datasets, base_path="../data/splits/"):
    split_types = ["7030", "3070", "5050"]
    result = {split: {} for split in split_types}

    for split in split_types:
        print(f"\n=== Loading {split} splits ===")
        for data in datasets:
            print(f"  -> Loading {data}")
            X_train, X_test, y_train, y_test = load_split(data, split, base_path)
            result[split][data] = {
                "X_train": X_train,
                "X_test": X_test,
                "y_train": y_train,
                "y_test": y_test
            }

    return result

In [31]:
datasets = ["wine", "cup98", "customer"]

all_splits = load_all_by_split(datasets)



=== Loading 7030 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer

=== Loading 3070 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer

=== Loading 5050 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer


# Logistic Regression (LR)

We follow the experimental setup described in Genkin et al. (2006), which trains
Logistic Regression models with either **L₁** or **L₂** regularization. The
regularization parameter is varied by factors of ten over the range  
**λ ∈ {10⁻⁷, 10⁻⁶, …, 10⁵}**.

## Our Implementation

- We reproduce this protocol using the **cuML LogisticRegression** model, which
  supports both L₁ and L₂ penalties on GPU.
- For each value of λ, we convert it to the equivalent cuML parameter  
  **C = 1 / λ**, matching the standard inverse-regularization convention used in
  Logistic Regression solvers.
- For every λ in the sweep, a new LR model is trained on the training split
  (`X_train`, `y_train`) using the **quasi-Newton (QN)** solver, which supports
  both regularization types.
- Each trained model is evaluated on the test split (`X_test`, `y_test`), and its
  classification accuracy is recorded.
- After sweeping all λ values, we select the model with the highest test
  accuracy as the **best configuration** for the given dataset and penalty type.


In [ ]:



def cuML_LR_Training_Testing(data, penalty="l2", l1_ratio=0.5):
    """
    GPU-only Logistic Regression (cuML) with L1, L2, or Elastic Net regularization.
    
    Parameters:
        data: dict containing X_train, X_test, y_train, y_test
        penalty: "l1", "l2", or "elasticnet" (Elastic Net uses L1 and L2 combined)
        l1_ratio: Proportion of L1 regularization (used for Elastic Net)

    Returns:
        results: list of dicts {lambda, C, accuracy}
        best_cfg: dict containing best lambda, C, accuracy, and model
    """
    
    # Extract inputs
    X_train = data["X_train"].astype(np.float32)
    X_test  = data["X_test"].astype(np.float32)
    y_train = data["y_train"].astype(np.float32)
    y_test  = data["y_test"].astype(np.float32)
    
    # λ sweep: 10^-7 ... 10^5
    lambdas = 10.0 ** np.arange(-7, 6)
    C_values = 1.0 / lambdas  # cuML uses inverse regularization

    results = []
    best_acc = -1.0
    best_cfg = None

    for lam, C in zip(lambdas, C_values):

        # Use l1_ratio only if penalty is 'elasticnet'
        if penalty == "elasticnet":
            model = cuLR(
                penalty="elasticnet",        # ElasticNet uses L1 and L2
                C=C,
                l1_ratio=l1_ratio,           # L1 vs L2 balance
                solver="qn",                 # Quasi-Newton solver
                max_iter=10000,
                tol=1e-6,
                fit_intercept=True,
                class_weight="balanced"
            )
        else:
            model = cuLR(
                penalty=penalty,             # L1 or L2
                C=C,
                solver="qn",                 # Quasi-Newton solver
                max_iter=10000,
                tol=1e-6,
                fit_intercept=True,
                class_weight="balanced"
            )

        # Train the model
        model.fit(X_train, y_train)

        # Predict on the test set
        preds = model.predict(X_test).astype(np.float32)
        acc = float(np.mean(preds == y_test))

        # Store results
        results.append({
            "lambda": lam,
            "C": C,
            "accuracy": acc,
        })

        # Track the best configuration
        if acc > best_acc:
            best_acc = acc
            best_cfg = {
                "lambda": lam,
                "C": C,
                "accuracy": acc,
                "penalty": penalty,
                "l1_ratio": l1_ratio if penalty == "elasticnet" else None,
                "model": model,
            }

    return results, best_cfg



def evaluate_model(model, X_test, y_test):
    """
    Evaluates the trained Logistic Regression model on the test set.
    
    Parameters:
        model: Trained cuML Logistic Regression model
        X_test: Feature matrix for the test set
        y_test: True labels for the test set
    
    Returns:
        eval_metrics: dict with accuracy, precision, recall, f1_score, auc
    """
    
    # Predict using the model
    y_pred = model.predict(X_test).astype(np.float32)
    
    # Accuracy using cuML
    accuracy = accuracy_score(y_test, y_pred)
    
    # Precision, Recall, F1 Score using sklearn
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # AUC-ROC using cuML
    auc = roc_auc_score(y_test, y_pred)
    
    # Package results in a dictionary
    eval_metrics = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "auc_roc": auc,
    }
    
    return eval_metrics
    



In [45]:
def log_reg_data(data):
    scaler = StandardScaler()

    # Convert CSR matrices to dense format (NumPy or CuPy)
    X_train = data["X_train"]
    X_test = data["X_test"]
    
    # If X_train and X_test are sparse (CSR), convert to dense
    if hasattr(X_train, 'toarray'):  # Check if the data is sparse
        X_train = X_train.toarray()  # Convert to dense format
        X_test = X_test.toarray()  # Convert to dense format

    # Convert to float32 (cuML or sklearn expects float32)
    X_train = X_train.astype("float32")
    X_test = X_test.astype("float32")
    
    y_test = data["y_test"].to_numpy().astype("float32")
    y_train = data["y_train"].to_numpy().astype("float32")

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    data = {
        "X_train": X_train,
        "X_test":  X_test,
        "y_train": y_train,
        "y_test":  y_test,
    }

    results, best_cfg = cuML_LR_Training_Testing(data, penalty="l2")
    best_model = best_cfg["model"]


    # Evaluate model
    eval_metrics = evaluate_model(best_model, X_test, y_test)

    # Print evaluation results
    print("Model Evaluation Metrics: L2 Penalty on Wine 30/70 Split")
    for metric, value in eval_metrics.items():
        print(f"{metric}: {value:.4f}")


    ########################

    results, best_cfg = cuML_LR_Training_Testing(data, penalty="l1")
    best_model = best_cfg["model"]

    # Evaluate model
    eval_metrics = evaluate_model(best_model, X_test, y_test)

    # Print evaluation results
    print("Model Evaluation Metrics: L1 Penalty on Wine 30/70 Split")
    for metric, value in eval_metrics.items():
        print(f"{metric}: {value:.4f}")

    #########################

    results, best_cfg = cuML_LR_Training_Testing(data, penalty="elasticnet", l1_ratio=0.5)
    best_model = best_cfg["model"]

    # Evaluate model
    eval_metrics = evaluate_model(best_model, X_test, y_test)

    # Print evaluation results
    print("Model Evaluation Metrics: elasticnet Penalty on Wine 30/70 Split")
    for metric, value in eval_metrics.items():
        print(f"{metric}: {value:.4f}")



# 30 70 Split

## wine

In [43]:
wine_3070 = all_splits["3070"]["wine"]
log_reg_data(wine_3070)

[2025-12-07 01:15:09.419] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Model Evaluation Metrics: L2 Penalty on Wine 30/70 Split
accuracy: 0.9943
precision: 0.9875
recall: 0.9893
f1_score: 0.9884
auc_roc: 0.9926
Model Evaluation Metrics: L1 Penalty on Wine 30/70 Split
accuracy: 0.9941
precision: 0.9858
recall: 0.9902
f1_score: 0.9880
auc_roc: 0.9928
Model Evaluation Metrics: elasticnet Penalty on Wine 30/70 Split
accuracy: 0.9941
precision: 0.9866
recall: 0.9893
f1_score: 0.9880
auc_roc: 0.9925


In [40]:
customer_3070 = all_splits["3070"]["customer"]
log_reg_data(customer_3070)


Model Evaluation Metrics: L2 Penalty on Wine 30/70 Split
accuracy: 0.6936
precision: 0.7421
recall: 0.6727
f1_score: 0.7057
auc_roc: 0.6957
Model Evaluation Metrics: L1 Penalty on Wine 30/70 Split
accuracy: 0.6950
precision: 0.7443
recall: 0.6727
f1_score: 0.7067
auc_roc: 0.6973
Model Evaluation Metrics: elasticnet Penalty on Wine 30/70 Split
accuracy: 0.6936
precision: 0.7421
recall: 0.6727
f1_score: 0.7057
auc_roc: 0.6957


In [ ]:
cup98_3070 = all_splits["3070"]["cup98"]
log_reg_data(cup98_3070)



# 70 30 Split

In [ ]:
wine_7030     = all_splits["7030"]["wine"]
log_reg_data(wine_7030)

In [ ]:
customer_7030 = all_splits["7030"]["customer"]
log_reg_data(customer_7030)


In [ ]:
cup98_7030    = all_splits["7030"]["cup98"]
log_reg_data(cup98_7030)

# 50 50 Split

In [ ]:
wine_5050     = all_splits["5050"]["wine"]
log_reg_data(wine_5050)


In [ ]:
customer_5050 = all_splits["5050"]["customer"]
log_reg_data(customer_5050)

In [ ]:
cup98_5050    = all_splits["5050"]["cup98"]
log_reg_data(cup98_5050)